# Notebook 03 — FastAPI 서빙

> DINO + SAM2 파이프라인을 REST API로 서빙

## 이 노트북에서 다루는 것

1. FastAPI 서버 구조 설명
2. 서버 실행 (uvicorn)
3. Python 클라이언트로 `/detect` 호출
4. 결과 이미지 시각화
5. Swagger UI 소개

---

## API 구조

```
POST /detect
  Request: { image_b64, prompt, box_threshold, use_sam }
  Response: { detections: [{label, confidence, box, has_mask}], elapsed_ms, image_b64 }

GET /health
  Response: { status: "ok" }
```

In [1]:
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import base64, io, time, subprocess, sys
import requests
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageDraw

IMAGE_DIR = Path('../data/test_images')
OUTPUT_DIR = Path('../output')
OUTPUT_DIR.mkdir(exist_ok=True)
API_URL = 'http://localhost:8000'

print('✅ 환경 설정 완료')
print(f'API URL: {API_URL}')

✅ 환경 설정 완료
API URL: http://localhost:8000


In [2]:
# 테스트 이미지 생성
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
img = Image.new('RGB', (640, 480), color=(135, 206, 235))
draw = ImageDraw.Draw(img)
draw.rectangle([0, 320, 640, 480], fill=(80, 80, 80))
draw.rectangle([80, 240, 280, 340], fill=(200, 50, 50))
draw.rectangle([100, 220, 260, 250], fill=(200, 50, 50))
draw.ellipse([420, 190, 460, 230], fill=(255, 220, 180))
draw.rectangle([415, 230, 465, 320], fill=(50, 50, 200))
draw.rectangle([540, 260, 560, 330], fill=(101, 67, 33))
draw.ellipse([510, 200, 590, 270], fill=(34, 139, 34))
img.save(IMAGE_DIR / 'street.jpg', 'JPEG')
print('✅ 테스트 이미지 준비')

✅ 테스트 이미지 준비


## 1. 서버 실행

**새 터미널**에서 아래 명령어 실행:

```bash
# grounding_pipeline 폴더에서
conda activate grounding_env
uvicorn app.main:app --reload --port 8000
```

서버가 뜨면 아래 셀을 실행하세요.

In [3]:
# 서버 헬스 체크
import requests

try:
    resp = requests.get(f'{API_URL}/health', timeout=3)
    print(f'✅ 서버 응답: {resp.json()}')
except requests.exceptions.ConnectionError:
    print('❌ 서버가 실행되지 않았습니다.')
    print('   터미널에서: uvicorn app.main:app --reload --port 8000')

✅ 서버 응답: {'status': 'ok'}


## 2. Python 클라이언트 함수

In [4]:
def image_to_b64(image_path):
    with open(image_path, 'rb') as f:
        return base64.b64encode(f.read()).decode()

def b64_to_image(b64_str):
    return Image.open(io.BytesIO(base64.b64decode(b64_str)))

def call_detect(image_path, prompt, threshold=0.25, use_sam=True):
    payload = {
        'image_b64': image_to_b64(image_path),
        'prompt': prompt,
        'box_threshold': threshold,
        'text_threshold': threshold,
        'use_sam': use_sam,
    }
    resp = requests.post(f'{API_URL}/detect', json=payload, timeout=60)
    resp.raise_for_status()
    return resp.json()

print('✅ 클라이언트 함수 정의 완료')

✅ 클라이언트 함수 정의 완료


## 3. API 호출 — DINO only (SAM 없이)

In [ ]:
print('DINO only 호출 중...')
t0 = time.time()
result = call_detect(
    IMAGE_DIR / 'street.jpg',
    prompt='person . tree . building.',
    threshold=0.2,
    use_sam=False
)
elapsed = time.time() - t0

print(f'\n응답 시간: {elapsed*1000:.0f}ms (서버 내부: {result["elapsed_ms"]}ms)')
print(f'감지된 객체: {len(result["detections"])}개')
for d in result['detections']:
    print(f'  - {d["label"]:20s} | score: {d["confidence"]:.3f} | box: {d["box"]}')

DINO only 호출 중...


## 4. API 호출 — DINO + SAM2 (마스크 포함)

> 첫 호출 시 모델 로드 시간 포함 (~30초)  
> 두 번째 호출부터는 빠릅니다

In [ ]:
print('DINO + SAM2 호출 중... (첫 호출은 모델 로드로 느림)')
t0 = time.time()
result = call_detect(
    IMAGE_DIR / 'street.jpg',
    prompt='person . tree . building.',
    threshold=0.2,
    use_sam=True
)
elapsed = time.time() - t0

print(f'\n응답 시간: {elapsed*1000:.0f}ms (서버 내부: {result["elapsed_ms"]}ms)')
print(f'감지된 객체: {len(result["detections"])}개')
for d in result['detections']:
    mask_str = '✅ 마스크' if d['has_mask'] else '  -'
    print(f'  {mask_str} {d["label"]:20s} | score: {d["confidence"]:.3f}')

# 결과 이미지 시각화
if result.get('image_b64'):
    result_img = b64_to_image(result['image_b64'])
    plt.figure(figsize=(10, 7))
    plt.imshow(result_img)
    plt.title('API 응답 — DINO + SAM2 결과 이미지')
    plt.axis('off')
    plt.tight_layout()
    out_path = OUTPUT_DIR / '03_api_result.jpg'
    result_img.save(out_path)
    plt.show()
    print(f'✅ 저장: {out_path}')
else:
    print('(결과 이미지 없음 — 감지 객체가 없거나 마스크 미생성)')

## 5. 응답속도 벤치마크 (웜업 후)

In [ ]:
N = 5
times = []

for i in range(N):
    t0 = time.time()
    r = call_detect(IMAGE_DIR / 'street.jpg', 'person . tree.', threshold=0.2, use_sam=False)
    times.append(r['elapsed_ms'])
    print(f'  [{i+1}/{N}] {r["elapsed_ms"]}ms')

print(f'\n평균: {sum(times)/len(times):.0f}ms')
print(f'최소: {min(times):.0f}ms')
print(f'최대: {max(times):.0f}ms')

plt.figure(figsize=(7, 4))
plt.bar(range(1, N+1), times, color='steelblue')
plt.axhline(sum(times)/len(times), color='tomato', linestyle='--',
            label=f'평균 {sum(times)/len(times):.0f}ms')
plt.xlabel('호출 횟수')
plt.ylabel('응답시간 (ms)')
plt.title('FastAPI /detect 응답속도 벤치마크')
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '03_api_benchmark.png', dpi=150)
plt.show()
print('✅ 저장: output/03_api_benchmark.png')

In [ ]:
print('=' * 52)
print('Notebook 03 — FastAPI 서빙 완료')
print('=' * 52)
print()
print('[학습한 것]')
print('  1. FastAPI로 DINO + SAM2 파이프라인 서빙')
print('  2. Base64 이미지 인코딩/디코딩')
print('  3. Python requests 클라이언트')
print('  4. API 응답속도 측정')
print()
print('[Swagger UI]')
print('  http://localhost:8000/docs  ← 브라우저에서 직접 테스트 가능')
print()
print('[저장된 결과물]')
print('  - output/03_api_result.jpg')
print('  - output/03_api_benchmark.png')
print()
print('[다음]')
print('  Docker 이미지 빌드 → GitHub 푸시')